In [2]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

pd.set_option("display.max_columns", None)

# Conexión a las bases de datos

In [ ]:
config = {
    'host': 'localhost',       
    'port': 
    'database': 'mensajeria',  
    'user': 'postgres',            
    'password':
}

In [3]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

    config_co = config['mensajeria_bd']
    config_etl = config['etl_mensajeria']

url_co = f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
url_etl = f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"


engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

# Extracción

In [4]:
df_novedad = pd.read_sql_table("mensajeria_novedadesservicio", engine_origen)

df_novedad.head()

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


In [5]:
df_novedad.info()

<class 'pandas.DataFrame'>
RangeIndex: 5208 entries, 0 to 5207
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   id               5208 non-null   int64              
 1   fecha_novedad    5208 non-null   datetime64[us, UTC]
 2   tipo_novedad_id  5208 non-null   int64              
 3   descripcion      5208 non-null   str                
 4   servicio_id      5208 non-null   int64              
 5   es_prueba        5208 non-null   bool               
 6   mensajero_id     5208 non-null   int64              
dtypes: bool(1), datetime64[us, UTC](1), int64(4), str(1)
memory usage: 249.3 KB


# Construcción de la dimensión tiempo

In [6]:
dim_tiempo = pd.DataFrame({
    "fecha": pd.date_range(
        start="2023-09-19",
        end="2024-08-31",
        freq="D"
    )
})

dim_tiempo["id_tiempo"] = range(1, len(dim_tiempo) + 1)

dim_tiempo.head()

,fecha,id_tiempo
0,2023-09-19,1
1,2023-09-20,2
2,2023-09-21,3
3,2023-09-22,4
4,2023-09-23,5


# Transformación de la tabla de hechos

In [7]:
fecha = (pd.to_datetime(df_novedad["fecha_novedad"]).dt.tz_localize(None).dt.normalize())

hecho = pd.DataFrame()

hecho["id_novedad_servicio"] = df_novedad["id"]

hecho["id_novedad"] = df_novedad["tipo_novedad_id"]

hecho["id_mensajero"] = df_novedad["mensajero_id"]

hecho["fecha"] = fecha

hecho = hecho.merge(dim_tiempo[["fecha", "id_tiempo"]], on="fecha", how="left")

hecho["cod_servicio"] = df_novedad["servicio_id"]

hecho["cantidad_novedades"] = 1

hecho.drop(columns=["fecha"], inplace=True)

hecho.head()

,id_novedad_servicio,id_novedad,id_mensajero,id_tiempo,cod_servicio,cantidad_novedades
0,4,1,7,73,51,1
1,5,1,7,73,51,1
2,6,1,7,73,51,1
3,7,1,7,73,51,1
4,8,1,7,73,51,1


# Validaciones

In [8]:
hecho.info()

<class 'pandas.DataFrame'>
RangeIndex: 5208 entries, 0 to 5207
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_novedad_servicio  5208 non-null   int64
 1   id_novedad           5208 non-null   int64
 2   id_mensajero         5208 non-null   int64
 3   id_tiempo            5208 non-null   int64
 4   cod_servicio         5208 non-null   int64
 5   cantidad_novedades   5208 non-null   int64
dtypes: int64(6)
memory usage: 244.3 KB


In [ ]:
hecho.isnull().sum()

In [12]:
# Debe devolver 0
hecho["id_tiempo"].isna().sum()

np.int64(0)